# DDM × Flexible PMC and RDM × Flexible PMC — ELPD comparison

Mirrors the Weber vs Flexible PMC comparison in `flexible_model_comparison.ipynb` but for the accumulator-model extensions added in Phase 5 of the cleanup/ddm-port branch.

Six labels per family (matching `submit_all_ddm_rdm_models.sh`):

- bare `ddm_flexible` / `rdm_flexible`: TMS on both perceptual and memory noise (paper's main claim).
- `_null`: no TMS regressor.
- `_perception`: TMS on perceptual noise only.
- `_memory`: TMS on memory noise only.
- `_threshold`: TMS on accumulator threshold `a` only (alternative-mechanism null).
- `_noise_threshold`: TMS on noise *and* threshold.

The threshold variants exist specifically so an ELPD argument can rule out a 'cTBS shifts caution' alternative.

In [ ]:
bids_folder = '/data/ds-tmsrisk/'
import os.path as op
import arviz as az
import pymc as pm
from tqdm.notebook import tqdm

from tms_risk.behavior.fit_model import build_model, get_data

In [ ]:
ddm_labels = [
    'ddm_flexible_null',
    'ddm_flexible_perception',
    'ddm_flexible_memory',
    'ddm_flexible',
    'ddm_flexible_threshold',
    'ddm_flexible_noise_threshold',
]
rdm_labels = [l.replace('ddm_', 'rdm_') for l in ddm_labels]
weber_pmc_labels = ['11_null', '11b', '11c', '11a']  # for cross-family reference
flex_pmc_labels = ['flexible2_null', 'flexible2b', 'flexible2a', 'flexible2']

all_labels = ddm_labels + rdm_labels + weber_pmc_labels + flex_pmc_labels

In [ ]:
idatas = {}
for label in tqdm(all_labels):
    path = op.join(bids_folder, 'derivatives', 'cogmodels', f'model-{label}_trace.netcdf')
    if not op.exists(path):
        print(f'MISSING: {label}')
        continue
    idatas[label] = az.from_netcdf(path)

print(f'Loaded {len(idatas)} / {len(all_labels)} traces')

In [ ]:
# Pretty names for the Table-1-style output
mapping = {
    # DDM family
    'ddm_flexible_null':            'DDM × Flexible PMC (null)',
    'ddm_flexible_perception':      'DDM × Flexible PMC (TMS on perceptual noise)',
    'ddm_flexible_memory':          'DDM × Flexible PMC (TMS on memory noise)',
    'ddm_flexible':                 'DDM × Flexible PMC (TMS on both noise terms)',
    'ddm_flexible_threshold':       'DDM × Flexible PMC (TMS on threshold)',
    'ddm_flexible_noise_threshold': 'DDM × Flexible PMC (TMS on noise + threshold)',
    # RDM family
    'rdm_flexible_null':            'RDM × Flexible PMC (null)',
    'rdm_flexible_perception':      'RDM × Flexible PMC (TMS on perceptual noise)',
    'rdm_flexible_memory':          'RDM × Flexible PMC (TMS on memory noise)',
    'rdm_flexible':                 'RDM × Flexible PMC (TMS on both noise terms)',
    'rdm_flexible_threshold':       'RDM × Flexible PMC (TMS on threshold)',
    'rdm_flexible_noise_threshold': 'RDM × Flexible PMC (TMS on noise + threshold)',
    # Reference: paper's choice-only models for context
    '11_null':       'Weber PMC (null)',
    '11b':           'Weber PMC (TMS on memory)',
    '11c':           'Weber PMC (TMS on perceptual)',
    '11a':           'Weber PMC (TMS on both)',
    'flexible2_null': 'Flexible PMC (null)',
    'flexible2b':     'Flexible PMC (TMS on perceptual only)',
    'flexible2a':     'Flexible PMC (TMS on memory only)',
    'flexible2':      'Flexible PMC (TMS on both)',
}

labeled = {mapping.get(k, k): v for k, v in idatas.items()}

In [ ]:
comparison_loo = az.compare(labeled, ic='loo')
comparison_loo

In [ ]:
az.plot_compare(comparison_loo, figsize=(8, 6))

In [ ]:
# Within-family rankings, for clarity
ddm_only = {k: v for k, v in labeled.items() if 'DDM ×' in k}
rdm_only = {k: v for k, v in labeled.items() if 'RDM ×' in k}
print('--- DDM family ---')
display(az.compare(ddm_only, ic='loo'))
print('--- RDM family ---')
display(az.compare(rdm_only, ic='loo'))

## What to look for

The paper's claim survives the DDM/RDM extension iff:

1. **`*_threshold` ranks below `*_perception` / `*_memory` / bare.** If TMS on threshold alone explained the choice + RT changes, the simpler caution-shift account would beat the noise account on ELPD.
2. **Within each family, the bare (TMS-on-both-noise) variant is close to or better than `*_null`.** Confirms the noise effect is detectable jointly with RT data, not just from choices.
3. **`*_noise_threshold` doesn't decisively beat the bare variant.** If it did, we'd need to add a caution-shift component to the paper's claim.

Cross-family: DDM/RDM should mirror the Weber → Flexible PMC ranking (flexible variants > Weber) seen in Table 1 of the manuscript.